# Multi-class Random Forest Classification — Feature Configuration I

Classifies allergenic tree crowns into five species: Betula, Alnus, Fraxinus,
Quercus, and Platanus.

**Input**: output of the binary classifier FC1
(`rf_binary/enschede_rf_binary_predictions_fc1_v01.shp`), which contains only crowns predicted
or confirmed as allergenic.

**Feature Configuration I** uses spectral and structural features only
(no NDVI). The original class distribution is preserved — Quercus is the
dominant class, which causes strong classification bias toward it. This
configuration is used as a baseline to demonstrate the effect of class
imbalance before the downsampling applied in FC2.

**Workflow**
1. Load binary predictions shapefile.
2. Split into labeled (public, species known) and unlabeled (private).
3. Prepare feature matrix — NDVI and binary prediction columns excluded.
4. Train/test split (80/20, stratified by species ID).
5. GridSearchCV with 10-fold cross-validation.
6. Evaluate on test set (accuracy, confusion matrix, per-species metrics).
7. Compute SHAP feature importance per species.
8. Predict species for private trees; apply 0.6 max-probability threshold.
9. Export to `rf_multiclass/enschede_rf_multiclass_predictions_fc1_v01.shp`.

**Import libraries**

In [ ]:
import time
import sys
from collections import Counter
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.paths import PROCESSED_DIR

**Define paths**

In [ ]:
# Output of the multiclass classifier FC1
BINARY_PRED_DIR = PROCESSED_DIR / "rf_binary/enschede_rf_binary_predictions_fc1_v01.shp"

OUTPUT_PATH = PROCESSED_DIR / "rf_multiclass/enschede_rf_multiclass_predictions_fc1_v01.shp"

**Define constants**

In [ ]:
CLASS_NAMES  = ["Betula", "Alnus", "Fraxinus", "Quercus", "Platanus"]
SPECIES_MAP  = {0: "Betula", 1: "Alnus", 2: "Fraxinus", 3: "Quercus", 4: "Platanus"}

In [ ]:
DROP_COLS_MC = ["id", "species_na", "is_allerge", "species_id",
                "geometry", "pred_is_al", "pred_prob_",
                "NDVI_mean", "NDVI_std"]

**Load data**

In [ ]:
trees_df = gpd.read_file(BINARY_PRED_DIR)
print(f"Total crowns: {len(trees_df)}")
print(trees_df["species_na"].value_counts(dropna=False))

**Split into labeled and unlabeled trees**

In [ ]:
# Labeled: public trees with a species_id (confirmed allergenic species)
# Unlabeled: private trees predicted as allergenic but without a species label
labeled_df   = trees_df[trees_df["species_id"].notna()].copy()
unlabeled_df = trees_df[trees_df["species_id"].isna()].copy()
print(f"Labeled:   {len(labeled_df)}")
print(f"Unlabeled: {len(unlabeled_df)}")

**Prepare feature matrices**

In [ ]:
X_labeled   = labeled_df.drop(columns=DROP_COLS_MC)
X_unlabeled = unlabeled_df.drop(columns=DROP_COLS_MC)
y_labeled   = labeled_df["species_id"]

print("X labeled shape:", X_labeled.shape)
print("Class distribution:")
print(labeled_df["species_na"].value_counts())

**Train/test split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_labeled, y_labeled,
    test_size=0.2,
    random_state=42,
    stratify=y_labeled,
)

**Train Random Forest with GridSearchCV**

In [ ]:
def train_rf(X_train, y_train, param_grid, label=""):
    """
    Train a Random Forest classifier using 10-fold GridSearchCV.

    Balanced class weights are applied to compensate for class imbalance.
    n_jobs=1 is used to ensure reproducibility and avoid memory issues on
    the CRIB computing platform.

    Args:
        X_train:    Training feature DataFrame.
        y_train:    Training target series.
        param_grid: Dict of hyperparameter ranges for GridSearchCV.
        label:      Optional label printed in the output for identification.

    Returns:
        The best fitted RandomForestClassifier from GridSearchCV.
    """
    classifier = RandomForestClassifier(
        random_state=42,
        n_jobs=1,
        class_weight="balanced",
    )
    grid = GridSearchCV(
        classifier,
        param_grid,
        cv=10,
        scoring="accuracy",
        n_jobs=1,
        verbose=1,
    )
    start = time.time()
    grid.fit(X_train, y_train)
    print(f"{label} Best parameters: {grid.best_params_}")
    print(f"{label} Best CV accuracy: {grid.best_score_:.4f}")
    print(f"{label} Training time:    {time.time() - start:.0f} s")
    return grid.best_estimator_

In [ ]:
param_grid = {
    "n_estimators": [200, 500, 800],
    "max_depth":    [None, 10, 20],
}

best_model = train_rf(X_train, y_train, param_grid, label="[FC1 Multi-class]")

**Evaluate on test set**

In [ ]:
def evaluate_multiclass(model, X_test, y_test, class_names):
    """
    Evaluate a multi-class Random Forest classifier on the test set.

    Prints accuracy, macro AUC-ROC, and a classification report.
    Plots a normalised confusion matrix and a per-species precision/recall chart.

    Args:
        model:       Fitted RandomForestClassifier.
        X_test:      Test feature DataFrame.
        y_test:      Test target series (integer class IDs).
        class_names: Ordered list of species name strings.
    """
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    print("Test accuracy:", accuracy_score(y_test, y_pred))
    print("Macro AUC-ROC:", roc_auc_score(y_test, y_prob, multi_class="ovr", average="macro"))
    print()
    print(classification_report(y_test, y_pred))

    # Normalised confusion matrix
    cm      = confusion_matrix(y_test, y_pred).T
    cm_norm = cm.astype("float") / cm.sum(axis=0)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, vmin=0, vmax=1)
    plt.title("Normalised Confusion Matrix")
    plt.xlabel("True")
    plt.ylabel("Predicted")
    plt.tight_layout()
    plt.show()

    # Per-species precision / recall bar chart
    report     = classification_report(y_test, y_pred, output_dict=True)
    class_keys = [k for k in report if k not in ("accuracy", "macro avg", "weighted avg")]
    df_m = pd.DataFrame({
        "Class":     [class_names[int(float(k))] for k in class_keys],
        "Precision": [report[k]["precision"] for k in class_keys],
        "Recall":    [report[k]["recall"]    for k in class_keys],
    })
    x  = np.arange(len(df_m))
    bw = 0.2
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, metric in enumerate(["Precision", "Recall"]):
        ax.bar(x + i * bw, df_m[metric], width=bw, label=metric)
    ax.set_xticks(x + bw / 2)
    ax.set_xticklabels(df_m["Class"], rotation=30, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Classification Metrics per Species")
    ax.legend()
    sns.despine()
    fig.tight_layout()
    plt.show()

In [ ]:
evaluate_multiclass(best_model, X_test, y_test, CLASS_NAMES)

**SHAP feature importance per species**

In [ ]:
def shap_analysis_multiclass(model, X_train, feature_names, class_names, n_samples=600):
    """
    Compute and plot SHAP feature importance for each species class.

    A random subsample of the training set is used. One horizontal bar chart
    is produced per species, arranged in a 2×3 grid layout.

    Args:
        model:         Fitted RandomForestClassifier.
        X_train:       Training feature DataFrame.
        feature_names: List of feature column names.
        class_names:   Ordered list of species name strings.
        n_samples:     Number of training samples to use for SHAP computation.
    """
    X_df     = pd.DataFrame(X_train, columns=feature_names)
    X_sample = X_df.sample(n_samples, random_state=42)

    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    fig = plt.figure(figsize=(16, 12), dpi=120)
    gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

    for i, sp in enumerate(class_names):
        if i < 3:
            ax = fig.add_subplot(gs[0, i])
        elif i == 3:
            ax = fig.add_subplot(gs[1, 0:2])
        else:
            ax = fig.add_subplot(gs[1, 2])

        shap_table = pd.DataFrame({
            "feature":      feature_names,
            "mean_abs_shap": np.abs(shap_values[i]).mean(axis=0),
        }).sort_values("mean_abs_shap", ascending=True).tail(15)

        colors = sns.color_palette("viridis", len(shap_table))
        bars   = ax.barh(shap_table["feature"], shap_table["mean_abs_shap"],
                         color=colors, edgecolor="black", alpha=0.8)
        ax.bar_label(bars,
                     labels=[f"{f}: {v:.4f}" for f, v in
                             zip(shap_table["feature"], shap_table["mean_abs_shap"])],
                     padding=3, fontsize=9, fontweight="bold")
        ax.set_title(sp, fontsize=14, weight="bold", pad=12)
        ax.set_xlabel("Mean Absolute SHAP Value", fontsize=11)
        ax.set_ylabel("Feature", fontsize=11)
        ax.set_xlim(0, shap_table["mean_abs_shap"].max() * 1.3)
        ax.grid(axis="x", linestyle="--", alpha=0.3)
        sns.despine(left=True, bottom=True)

    fig.suptitle("Top 15 Features by SHAP Importance per Species",
                 fontsize=16, weight="bold", y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
shap_analysis_multiclass(best_model, X_train, list(X_labeled.columns), CLASS_NAMES)

**Predict species for private trees**

In [ ]:
# Predict species for private trees (unlabeled dataset).
prob_cols = [f"prob_{s}" for s in CLASS_NAMES]

unlabeled_df["pred_species"] = best_model.predict(X_unlabeled)
unlabeled_df[prob_cols]      = best_model.predict_proba(X_unlabeled)

print("Private tree species distribution:")
print(unlabeled_df["pred_species"].map(SPECIES_MAP).value_counts())

**Filter and export predictions**

In [ ]:
# Write predictions back onto the full GeoDataFrame
for col in prob_cols:
    trees_df[col] = np.nan
trees_df.loc[unlabeled_df.index, prob_cols]        = unlabeled_df[prob_cols].values
trees_df.loc[unlabeled_df.index, "pred_species"]   = unlabeled_df["pred_species"].values

# Maximum class probability — used as confidence filter for private trees
trees_df["pred_prob_max"] = trees_df[prob_cols].max(axis=1)

# Assign species name:
#   - public trees: use true species label
#   - private trees: map predicted class ID to species name
trees_df["full_speci"] = trees_df["pred_species"].map(SPECIES_MAP)
trees_df.loc[trees_df["species_id"].notna(), "full_speci"] = trees_df["species_na"]

# Keep public allergenic trees and private trees with confidence >= 0.6
trees_filtered = trees_df[
    (trees_df["is_allerge"] == 1) |
    (trees_df["is_allerge"].isna() & (trees_df["pred_prob_max"] >= 0.6))
].copy()

# Drop columns that are not needed downstream
trees_filtered = trees_filtered.drop(
    columns=["is_allerge", "species_id", "pred_is_al", "pred_prob_",
             "pred_species"] + prob_cols,
    errors="ignore",
)

print(f"Total retained: {len(trees_filtered)}")
trees_filtered.to_file(OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH}")